<div dir="rtl">
<h1>درخواست را ببینید، پاسخ را نمره بدهید</h1>
<p>درس 73 از 76 · آموزش روی دستور چه چیزی را تغییر می‌دهد؟ · <code dir="ltr">65-sft</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-02/65-sft.html">📖 بازگشت به همین درس</a></p>
<p>Loss پاسخ را بیرون API فعلی MiniGPT بسازید و مرز آن را با Causal Mask جدا کنید.</p><p>پیش‌نیاز: شیفت یک‌خانه‌ای Target و Cross-Entropy با Reduction='none'.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>در [q0,q1,sep,a0,a1]، آیا خروجی موقعیت sep باید در Loss پاسخ حساب شود؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from torch.nn import functional as F
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(31)
model = MiniGPT(ModelConfig(8,4,8,2,1,0.0))
x,targets = torch.tensor([[1,2,3,4]]),torch.tensor([[2,3,4,5]])
selected = torch.tensor([[False,False,True,True]])
logits,_ = model(x)
print('logits shape:',tuple(logits.shape))
print('target ID / counted as response:',list(zip(targets[0].tolist(),selected[0].tolist())))
print('all targets:',targets.numel(),'response targets:',selected.sum().item())

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>response_loss(Logits,targets,mask) فقط Cross-Entropy هدف‌هایی با mask=True را میانگین بگیرد. Logits شکل (B,T,V)، دو ورودی دیگر شکل (B,T) دارند. Mask خالی از True باید ValueError بدهد؛ از Loss آمادهٔ model(x,targets) استفاده نکنید چون قبلاً میانگین گرفته شده است.</p>
</div>

In [ ]:
def response_loss(logits, targets, mask):
    # TODO: اول Loss هر Target، سپس انتخاب و میانگین
    return None

In [ ]:
def test_exercise():
    result = response_loss(logits,targets,selected)
    if result is None:
        return False
    expected_response = F.cross_entropy(logits[0,2:,:],targets[0,2:])
    torch.testing.assert_close(result,expected_response)
    changed = targets.clone()
    changed[0,:2] = torch.tensor([6,7])
    torch.testing.assert_close(result,response_loss(logits,changed,selected))
    expected_all = F.cross_entropy(logits[0],targets[0])
    torch.testing.assert_close(response_loss(logits,targets,torch.ones_like(selected)),expected_all)
    try:
        response_loss(logits,targets,torch.zeros_like(selected))
    except ValueError:
        pass
    else:
        raise AssertionError('no selected targets')
    model.zero_grad(set_to_none=True)
    fresh_logits,_ = model(x)
    response_loss(fresh_logits,targets,selected).backward()
    assert model.token_embedding.weight.grad[1].abs().sum()>0
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: response_loss')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط تعداد موقعیت‌های درخواست را در یک ماسک نمونه بیشتر کنید؛ دو موقعیت پاسخ ثابت بمانند. تعداد کل موقعیت‌ها و تعداد Trueها چه تغییری می‌کنند؟ برای میانگین Loss پاسخ، کدام تعداد باید مخرج باشد؟ این آزمایش فقط شمارش موقعیت‌هاست، نه اجرای مدل با درخواست بلندتر.</p>
</div>

In [ ]:
response_mask = torch.tensor([[False,False,True,True]])
for extra_prompt_positions in (0,3,8):
    prefix = torch.zeros((1,extra_prompt_positions),dtype=torch.bool)
    longer_mask = torch.cat((prefix,response_mask),dim=1)
    print('extra prompt positions:',extra_prompt_positions,
          'all positions:',longer_mask.numel(),
          'response positions:',torch.count_nonzero(longer_mask).item())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>ماسکِ بخش ورودی، نخستین Token پاسخ را از Loss جا می‌اندازد. target_response_mask(roles) برای نقش‌های کل دنباله، ماسک Targetهای شیفت‌یافته را بدهد؛ فقط نقش 'answer' انتخاب می‌شود.</p>
</div>

In [ ]:
roles = ['prompt','prompt','separator','answer','answer']
wrong = [role=='answer' for role in roles[:-1]]
print('input-based mask:',wrong)
print('input/target roles:',list(zip(roles[:-1],roles[1:])))

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def target_response_mask(roles):
    # TODO: نقش هر Target، نه نقش ورودی همان موقعیت
    return None

In [ ]:
def test_repair():
    result = target_response_mask(roles)
    if result is None:
        return False
    assert result==[False,False,True,True]
    assert target_response_mask(['separator','answer'])==[True]
    assert target_response_mask(['prompt','separator'])==[False]
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: target_response_mask')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>forward واقعی برای Logits استفاده شد؛ Response mask به‌صورت آموزشی بیرون آن محاسبه شد. API فعلی Target منفی را رد می‌کند و SFT آماده ندارد. حذف Loss درخواست، مسیر Gradient از پاسخ به نمایش درخواست را حذف نمی‌کند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا حذف یک موقعیت از میانگین Loss با منع Attention به آن موقعیت فرق دارد؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-02/65-sft.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/65-sft.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>